In [1]:
import duckdb
from pathlib import Path

# Project paths
project_dir = Path("..")
data_path = project_dir / "data" / "cleaned_telco_churn.csv"

# Connect to DuckDB
con = duckdb.connect()

print("DuckDB connected successfully!")
print("Data path:", data_path)

DuckDB connected successfully!
Data path: ..\data\cleaned_telco_churn.csv


In [2]:
query = f"""
SELECT
    COUNT(*) AS total_customers,
    SUM(Churn) AS churned_customers,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2) AS churn_rate
FROM read_csv_auto('{data_path.as_posix()}');
"""

result = con.execute(query).df()

result

,total_customers,churned_customers,churn_rate
0,7043,1869.0,26.54


In [3]:
query = f"""
SELECT
    Contract,
    COUNT(*) AS total_customers,
    SUM(Churn) AS churned_customers,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2) AS churn_rate
FROM read_csv_auto('{data_path.as_posix()}')
GROUP BY Contract
ORDER BY churn_rate DESC;
"""

contract_result = con.execute(query).df()

contract_result

,Contract,total_customers,churned_customers,churn_rate
0,Month-to-month,3875,1655.0,42.71
1,One year,1473,166.0,11.27
2,Two year,1695,48.0,2.83


In [4]:
query = f"""
SELECT
    InternetService,
    COUNT(*) AS total_customers,
    SUM(Churn) AS churned_customers,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2) AS churn_rate
FROM read_csv_auto('{data_path.as_posix()}')
GROUP BY InternetService
ORDER BY churn_rate DESC;
"""

internet_result = con.execute(query).df()

internet_result


,InternetService,total_customers,churned_customers,churn_rate
0,Fiber optic,3096,1297.0,41.89
1,DSL,2421,459.0,18.96
2,No,1526,113.0,7.40


In [5]:
query = f"""
SELECT
    customerID,
    tenure,
    Contract,
    InternetService,
    MonthlyCharges,
    TotalCharges
FROM read_csv_auto('{data_path.as_posix()}')
WHERE Churn = 1
  AND MonthlyCharges >= 80
ORDER BY MonthlyCharges DESC;
"""

high_value_churn = con.execute(query).df()

high_value_churn.head(20)

,customerID,tenure,Contract,InternetService,MonthlyCharges,TotalCharges
0,8199-ZLLSA,67,One year,Fiber optic,118.35,7804.15
1,2889-FPWRM,72,One year,Fiber optic,117.80,8684.80
2,2302-ANTDP,48,Month-to-month,Fiber optic,117.45,5438.90
3,9053-JZFKV,67,Two year,Fiber optic,116.20,7752.30
4,1444-VVSGW,70,One year,Fiber optic,115.65,7968.85
5,0201-OAMXR,70,One year,Fiber optic,115.55,8127.60
6,4361-BKAXE,41,Month-to-month,Fiber optic,114.50,4527.45
7,1555-DJEQW,70,Two year,Fiber optic,114.20,7723.90
8,9158-VCTQB,41,Month-to-month,Fiber optic,113.60,4594.95
9,7279-BUYWN,41,Month-to-month,Fiber optic,113.20,4689.50


In [6]:
query = f"""
SELECT
    CASE
        WHEN tenure <= 12 THEN '0-12 months'
        WHEN tenure <= 24 THEN '13-24 months'
        WHEN tenure <= 48 THEN '25-48 months'
        WHEN tenure <= 60 THEN '49-60 months'
        ELSE '61-72 months'
    END AS tenure_group,

    COUNT(*) AS total_customers,
    SUM(Churn) AS churned_customers,

    ROUND(
        100.0 * SUM(Churn) / COUNT(*),
        2
    ) AS churn_rate

FROM read_csv_auto('{data_path.as_posix()}')

GROUP BY tenure_group

ORDER BY
    CASE tenure_group
        WHEN '0-12 months' THEN 1
        WHEN '13-24 months' THEN 2
        WHEN '25-48 months' THEN 3
        WHEN '49-60 months' THEN 4
        WHEN '61-72 months' THEN 5
    END;
"""

tenure_result = con.execute(query).df()

tenure_result

,tenure_group,total_customers,churned_customers,churn_rate
0,0-12 months,2186,1037.0,47.44
1,13-24 months,1024,294.0,28.71
2,25-48 months,1594,325.0,20.39
3,49-60 months,832,120.0,14.42
4,61-72 months,1407,93.0,6.61


# Day 3 SQL Analysis — Key Findings

### 1. Overall Churn
The overall customer churn rate was approximately 26.54%.

### 2. Contract Type
The highest churn rate was observed among month-to-month contract customers.

### 3. Internet Service
The fiber optic internet service category showed the highest churn rate.

### 4. High-Value Customers
A significant number of churned customers had monthly charges above $80, indicating potential high-value revenue loss.

### 5. Tenure
Customers in the 0 yo 12 months tenure group showed the highest churn rate.